# Demo on Training and Evaluating RF Surrogate Model from Scratch

###  Introduction
This notebook demonstrates how to train a RandomForest (RF) for performance of parametrized quantum circuits (PQCs).
The circuits are first converted into **Tensors** that capture structural and gate-based information.

### Imports and Setup

First, we suppress PyTorch Warnings and ensure proper dependency loading. Then, we import core modules and helper functions from our project structure.

In [1]:
# Suppress PyTorch Warnings and ensure proper dependency loading
import warnings
import os
import sys

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) # Go two levels up from the notebook location to reach project root

if project_root not in sys.path:
    sys.path.insert(0, project_root)
print("Project root set to:", project_root)

Project root set to: /Users/danielbarta/Desktop/work/FOKUS/SQuASH


In [2]:
import os
import json
import torch

from surrogate_models.architectures.random_forest.random_forest_runner import set_seed, prepare_paths_and_config, prepare_dataset, prepare_dataloaders, save_json, train_model, save_results, plot_metrics

from surrogate_models.architectures.random_forest.random_forest_runner import train_model
from surrogate_models.architectures.random_forest.random_forest_runner import evaluate_model

### 📁 Step 1: Prepare Paths and Configuration


The configuration for the surrogate model is defined and loaded in two stages:
 - setting the search space and device
 - specify config for the surrogate model

In [3]:
search_space = 'ghz_a'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

This sets the quantum circuit benchmark domain (search_space) and selects the device (cpu or cuda) for training.


In [4]:
config, gate_set, timestamp = prepare_paths_and_config(search_space, device)
print("Config (incl. model config):")
print(json.dumps(config, indent=4, default=str))



Config (incl. model config):
{
    "device": "cpu",
    "seed": 42,
    "runseed": 42,
    "batch_size": 64,
    "num_workers": 0,
    "epochs": null,
    "emb_dim": null,
    "layer_num": null,
    "qubit_num": null,
    "num_node_features": 7,
    "drop_ratio": null,
    "lr": null,
    "decay": null,
    "JK": null,
    "patience": 7,
    "metric": "mse",
    "graph_pooling": null,
    "n_estimators": 350,
    "max_depth": 30,
    "random_state": null,
    "optuna_trials": null,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "n_jobs": -1,
    "PATHS": {
        "optuna_studies": "/Users/danielbarta/Desktop/work/FOKUS/SQuASH/surrogate_models/tuning/studies",
        "raw_data": "/Users/danielbarta/Desktop/work/FOKUS/SQuASH/data/raw_data/",
        "gcn_data": "/Users/danielbarta/Desktop/work/FOKUS/SQuASH/data/processed_data/gcn_processed_data",
        "rf_data": "/Users/danielbarta/Desktop/work/FOKUS/SQuASH/data/processed_data/rf_processed_dat

### 🔧 Step 2: Set Random Seed

In [5]:
set_seed(config["runseed"])

### 📦 Step 3: Load or Generate Dataset

To be processed by RF, the data—i.e. quantum circuits—need to be transformed into tensors. If preprocessed data is not found, it's automatically created from raw quantum circuit files.

In [6]:
data_name = f"rf_demo_dataset_ghz_a"
data_path = os.path.join(config['PATHS']['rf_data'], f'{data_name}.pt')
train_data, val_data, test_data = prepare_dataset(data_name, data_path, gate_set, search_space, config, timestamp)

Data successfully loaded from /Users/danielbarta/Desktop/work/FOKUS/SQuASH/data/processed_data/rf_processed_data/rf_demo_dataset_ghz_a.pt
[INFO] Train: 13894, Val: 2605, Test: 869
Successfully saved to /Users/danielbarta/Desktop/work/FOKUS/SQuASH/data/processed_data/rf_processed_data/test_rf_ghz_a_2025-05-30_12-31-02.pt: 869 items


### 🧪 Step 4: Prepare DataLoaders

We use PyTorch Geometric's `DataLoader` to batch and feed our graph data efficiently into the RandomForest.

In [7]:
train_loader, val_loader, test_loader = prepare_dataloaders(
    train_data, val_data, test_data,
    batch_size=config['batch_size'],
    num_workers=config['num_workers']
)

### 📝 Step 5: Save Model Configuration

We persist the configuration used for this training session for reproducibility.

In [8]:
model_name = f"demo_rf_{search_space}_{timestamp}"
model_logs_path = os.path.join(config['PATHS'][f'trained_models'], f'{model_name}')
os.makedirs(model_logs_path, exist_ok=True)
save_json(os.path.join(model_logs_path, f'{model_name}_config.json'), config)

### 🚀 Step 6: Train the RF Model
We use a `RandomForestRegressor` from `sklearn`.

In [9]:
print("[INFO] Starting training")
rf_model = train_model(config, train_data)
print(rf_model)

[INFO] Starting training
RandomForestRegressor(max_depth=30, max_features='sqrt', n_estimators=350,
                      n_jobs=-1, random_state=42)


### 📊 Step 7: Evaluate Model Performance

We inspect the best achieved validation metric and performance on the test set.


In [10]:
# Evaluate the model on the validation set.
print("[INFO] Evaluating on Validation Set")
val_mse, val_rho, val_r2 = evaluate_model(rf_model, val_data, desc="Validation")

# Evaluate the model on the test set.
print("[INFO] Evaluating on Test Set")
test_mse, test_rho, test_r2 = evaluate_model(rf_model, test_data, desc="Testing")


[INFO] Evaluating on Validation Set
[Validation] MSE=0.0167, R2=0.3358, Spearman=0.6615
[INFO] Evaluating on Test Set
[Testing] MSE=0.0166, R2=0.3539, Spearman=0.6729
